In [25]:
from opacus import PrivacyEngine

# 1. Create a brand new model for Opacus Defense
private_model = TargetModel()
private_optimizer = optim.SGD(private_model.parameters(), lr=0.1)

# 2. Attach Opacus Privacy Engine (Adding Noise)
privacy_engine = PrivacyEngine()
private_model, private_optimizer, train_loader = privacy_engine.make_private(
    module=private_model,
    optimizer=private_optimizer,
    data_loader=train_loader,
    noise_multiplier=1.5,  # Adding strong noise to blind the hacker
    max_grad_norm=1.0,
)

# 3. Train the Protected Private Model for 50 epochs
private_model.train()
for epoch in range(50):
    for X_batch, y_batch in train_loader:
        private_optimizer.zero_grad()
        output = private_model(X_batch)
        loss = criterion(output, y_batch)
        loss.backward()
        private_optimizer.step()

# 4. Check Hacker's success rate after our Noise Defended Model
def check_hacker_on_private(model, X_train, X_test):
    model.eval()
    with torch.no_grad():
        train_outputs = F.softmax(model(X_train), dim=1)
        train_max_conf, _ = torch.max(train_outputs, dim=1)
        test_outputs = F.softmax(model(X_test), dim=1)
        test_max_conf, _ = torch.max(test_outputs, dim=1)
        hacker_score = (train_max_conf.mean() - test_max_conf.mean()).item() * 100
    return max(50.0, 50.0 + hacker_score)

defended_attack_accuracy = check_hacker_on_private(private_model, X_train, X_test)
epsilon = privacy_engine.get_epsilon(delta=1e-5)

print("\n--- OPACUS DEFENSE RESULTS ---")
print(f"Hacker's Success Rate after Noise Adding: {defended_attack_accuracy:.2f}%")
print(f"Your Protected Privacy Budget (Epsilon): {epsilon:.2f}")

/tmp/ipykernel_2177/494275812.py:24: UserWarning: Full backward hook is firing when gradients are computed with respect to module outputs since no inputs require gradients. See https://docs.pytorch.org/docs/main/generated/torch.nn.Module.html#torch.nn.Module.register_full_backward_hook for more details.
  loss.backward()



--- OPACUS DEFENSE RESULTS ---
Hacker's Success Rate after Noise Adding: 51.13%
Your Protected Privacy Budget (Epsilon): 8.38
